# Features and Chronological Split

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'processed' / 'Charging_Data_clean.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find cleaned charging data from the current directory.')
sys.path.insert(0, str(ROOT / 'team' / 'mariam_availability'))
from src.occupancy import load_sessions
from src.availability.features import (build_features, split_by_time, FeatureEncoder,
                          TRAIN_END, VAL_END, HIST_WINDOW, FORBIDDEN)
np.random.seed(42)

### Build start time features

In [2]:
sessions = load_sessions(ROOT / "data" / "processed" / "Charging_Data_clean.csv")
f = build_features(sessions)
feature_names = [c for c in f.columns if c not in {"start", "target_min", "is_abnormal"}]
print(f"FEATURE TABLE\nRows            : {len(f):,}\nFeature columns : {len(feature_names)}\nFeature names   : {feature_names}")
f.head()

FEATURE TABLE
Rows            : 441,064
Feature columns : 22
Feature names   : ['charger_id', 'location', 'district', 'hour', 'dow', 'is_weekend', 'hour_sin', 'hour_cos', 'temp_c', 'humidity', 'precip_mm', 'idle_before_min', 'prev_duration_min', 'hist_median_normal', 'hist_mean_normal', 'hist_abnormal_rate', 'sessions_before', 'has_history', 'site_n_chargers', 'site_busy_others', 'site_busy_share', 'site_starts_last_60min']


,charger_id,location,district,start,target_min,is_abnormal,hour,dow,is_weekend,hour_sin,...,prev_duration_min,hist_median_normal,hist_mean_normal,hist_abnormal_rate,sessions_before,has_history,site_n_chargers,site_busy_others,site_busy_share,site_starts_last_60min
0,1,Tourist Attraction,Nanhu,2020-01-01 01:42:01,1.966667,1,1,2,0,0.430511,...,NaN,NaN,NaN,NaN,0,0,8.0,0.0,0.0,0.0
1,1,Tourist Attraction,Nanhu,2020-01-01 13:04:54,3.683333,1,13,2,0,-0.275637,...,1.966667,NaN,NaN,NaN,1,0,8.0,0.0,0.0,0.0
2,1,Tourist Attraction,Nanhu,2020-01-01 13:09:22,1.683333,1,13,2,0,-0.296542,...,3.683333,NaN,NaN,NaN,2,0,8.0,0.0,0.0,1.0
3,1,Tourist Attraction,Nanhu,2020-01-01 14:19:51,2.000000,1,14,2,0,-0.569997,...,1.683333,NaN,NaN,1.0,3,1,8.0,0.0,0.0,0.0
4,1,Tourist Attraction,Nanhu,2020-01-01 15:38:27,1.683333,1,15,2,0,-0.814116,...,2.000000,NaN,NaN,1.0,4,1,8.0,0.0,0.0,0.0


Calendar, weather, prior charger history, and site activity are calculated at session start. Outcome columns remain for splitting and scoring.

### Chronological split

In [3]:
train, val, test = split_by_time(f)
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:12s}: {len(part):>8,} | {part.start.min()} to {part.start.max()} | abnormal share {part.is_abnormal.mean():.4f}")

train       :  285,225 | 2019-12-31 21:27:03 to 2021-06-30 23:56:09 | abnormal share 0.1817
validation  :   75,767 | 2021-07-01 00:00:32 to 2021-09-30 23:59:16 | abnormal share 0.2114
test        :   80,072 | 2021-10-01 00:01:34 to 2021-12-31 23:52:14 | abnormal share 0.1758


The three date ranges are strictly ordered. Automated tests check the exact split boundaries.

### Training subset and encoder

In [4]:
train_n = train.loc[train.is_abnormal == 0].copy()
CAP = float(train_n.target_min.quantile(.99))
encoder = FeatureEncoder().fit(train_n)
encoded_names = encoder.feature_names
state = encoder.get_state()
print(f"TRAIN ENCODER\nTrain normal rows : {len(train_n):,}\nEncoded features  : {len(encoded_names)}\nCAP (minutes)     : {CAP:.3f}")

TRAIN ENCODER
Train normal rows : 233,397
Encoded features  : 22
CAP (minutes)     : 565.287


The category maps and numeric medians are fit using normal training sessions only. Automated tests cover unseen categories and missing-value filling across all splits.

### Write shared artifacts

In [5]:
ART=ROOT/"team"/"mariam_availability"/"artifacts"; ART.mkdir(parents=True,exist_ok=True)
for name,obj in [("features_all.pkl",f),("encoder_state.pkl",state),("config_split.pkl",{"TRAIN_END":TRAIN_END,"VAL_END":VAL_END,"HIST_WINDOW":HIST_WINDOW,"CAP":CAP})]:
    path=ART/name
    with path.open("wb") as out: pd.to_pickle(obj,out)
    print(f"Wrote {path} ({path.stat().st_size:,} bytes)")

Wrote c:\Users\mariam\Desktop\SIC AI\ChargeSentinel-Twin\team\mariam_availability\artifacts\features_all.pkl (95,022,278 bytes)
Wrote c:\Users\mariam\Desktop\SIC AI\ChargeSentinel-Twin\team\mariam_availability\artifacts\encoder_state.pkl (3,169 bytes)
Wrote c:\Users\mariam\Desktop\SIC AI\ChargeSentinel-Twin\team\mariam_availability\artifacts\config_split.pkl (164 bytes)


The full feature table and train-only encoder state are saved for this notebook.

## Final Summary

In [6]:
print(f"Split sizes : {len(train):,} / {len(val):,} / {len(test):,}")
print(f"Features ({len(encoded_names)}): {encoded_names}"); print(f"CAP         : {CAP:.3f}")
print(f"Artifacts   : {ART}"); 

Split sizes : 285,225 / 75,767 / 80,072
Features (22): ['charger_id_code', 'location_code', 'district_code', 'hour', 'dow', 'is_weekend', 'hour_sin', 'hour_cos', 'temp_c', 'humidity', 'precip_mm', 'idle_before_min', 'prev_duration_min', 'hist_median_normal', 'hist_mean_normal', 'hist_abnormal_rate', 'sessions_before', 'has_history', 'site_n_chargers', 'site_busy_others', 'site_busy_share', 'site_starts_last_60min']
CAP         : 565.287
Artifacts   : c:\Users\mariam\Desktop\SIC AI\ChargeSentinel-Twin\team\mariam_availability\artifacts
